# BRONZE → SILVER: Transformacion de Customers

Este notebook ejecuta el script de transformacion que:
1. Lee la capa Bronze (Parquet)
2. Aplica limpieza: trim de strings, deduplicacion, filtro de nulos, email a minusculas
3. Agrega columnas de metadatos
4. Guarda los datos limpios en la capa Silver (Parquet)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

## 1. Verificar que la capa Bronze existe

In [ ]:
BRONZE_PATH = PROJECT_ROOT / 'data' / 'bronze' / 'customers'

if not BRONZE_PATH.exists():
    raise FileNotFoundError(
        f'Bronze layer no encontrada en {BRONZE_PATH}.\n'
        'Ejecuta primero el notebook 01_raw_to_bronze_customers.ipynb'
    )
print(f'Bronze layer encontrada en: {BRONZE_PATH}')

## 2. Ejecutar el script BRONZE → SILVER

In [ ]:
from src.bronze_to_silver.customers_transform import transform_customers

total_records = transform_customers()
print(f'\nTransformacion completada: {total_records} registros en Silver.')

## 3. Validar la capa Silver y revisar cuarentena

In [ ]:
from src.utils.spark_session import create_spark_session

SILVER_PATH = PROJECT_ROOT / 'data' / 'silver' / 'customers'

spark = create_spark_session('notebook_silver_validation')
df_silver = spark.read.parquet(str(SILVER_PATH))

print('Schema Silver:')
df_silver.printSchema()

In [ ]:
print(f'Total registros Silver: {df_silver.count()}')
df_silver.show(10, truncate=False)

In [ ]:
# Verificar que no hay nulos en customer_id (todas las filas deben haber pasado DQX)
from pyspark.sql import functions as F

null_count = df_silver.filter(F.col('customer_id').isNull()).count()
print(f'Registros con customer_id nulo: {null_count}')  # debe ser 0

# Revisar capa de cuarentena (filas que fallaron las reglas DQX)
QUARANTINE_PATH = PROJECT_ROOT / 'data' / 'quarantine' / 'customers'

if QUARANTINE_PATH.exists():
    df_quarantine = spark.read.parquet(str(QUARANTINE_PATH))
    print(f'\nRegistros en cuarentena: {df_quarantine.count()}')
    print('\nColumnas de audit DQX (_errors, _warnings):')
    df_quarantine.select('customer_id', 'nombre', 'email', '_errors', '_warnings').show(truncate=False)
else:
    print('\nNo se generó capa de cuarentena — todos los registros pasaron las validaciones DQX.')

In [ ]:
spark.stop()
print('SparkSession cerrada.')